<div dir="rtl" style="text-align: right;">

# Queuechella — סימולציה של פסטיבל מוזיקה

**קורס סימולציה, סמסטר ב' 2026 — אוניברסיטת בן גוריון**

סימולציית אירועים בדידים (DES) של פסטיבל מוזיקה בן יומיים. במודל שלושה סוגי מבקרים (`FriendsGroup`, `Couple`, `Single`), שש תחנות שירות (FIFO עם מספר שרתים) ושלוש במות עם קיבולת מוגבלת.

**מטרת הניתוח**: להשוות שלוש חלופות שדרוג מול תרחיש בסיס במסגרת תקרת תקציב של 1,000,000 ₪.

מבנה הנוטבוק תואם את הפרויקט לדוגמה (HotelSimulation):
1. התאמת התפלגויות לנתוני האקסל
2. ניתוח זמן חימום (Welch moving average)
3. תרחיש בסיס — Data Points For Current State
4. סקירת חלופות — Alternative Review
5. השוואת חלופות במבחן Welch
6. מסקנות והמלצות

</div>

<div dir="rtl" style="text-align: right;">

## הכנת סביבת הריצה

מודולי הסימולציה (`engine`, `sim_stats`, `distributions` וכו') כבר כתובים כקבצי `.py` בפרויקט. הנוטבוק מייבא אותם ישירות — זה עובד גם ב-VS Code (כשהנוטבוק יושב בתיקיית הפרויקט) וגם ב-Colab אחרי שמשכפלים את הריפו.

</div>

In [ ]:
# When running on Colab: clone the repo and switch into the project folder
import sys, os
if 'google.colab' in sys.modules:
    if not os.path.isdir('simu_v2'):
        !git clone https://github.com/saaragmon/simu_v2.git
    os.chdir('simu_v2')
    !pip install -q scipy openpyxl matplotlib

sys.path.insert(0, os.getcwd())
print('cwd:', os.getcwd())

In [ ]:
# imports
import time
import math
import statistics as stats_lib
import matplotlib.pyplot as plt
from scipy.stats import t as student_t

from config import SimConfig
from engine import Simulation
from sim_stats import MultiRunStatistics, RunStatistics
from alternatives import build_baseline, build_combo_a, build_combo_b, build_combo_c
from distributions import fit_from_excel
from warmup import WarmupSimulation

# Global parameters (kept in sync with main.py)
EXCEL_PATH = 'samples_for_simulation.xlsx'
NUM_RUNS = 20
CONFIDENCE_LEVEL = 0.90
RELATIVE_PRECISION = 0.10

KPIS = ['avg_satisfaction', 'total_revenue', 'avg_queue_length']
KPI_HIGHER_IS_BETTER = {
    'avg_satisfaction':   True,
    'total_revenue':      True,
    'avg_queue_length':   False,
}

<div dir="rtl" style="text-align: right;">

## 1. התאמת התפלגויות לנתוני האקסל (Goodness-of-Fit)

באקסל `samples_for_simulation.xlsx` שתי עמודות:
- **גיליון 1**: זמני הגעה בין-`FriendsGroup`
- **גיליון 2**: משכי הופעה במה ראשית

לכל עמודה מתבצעת התאמה לשלוש משפחות מועמדות (אקספוננציאלית, נורמלית, אחידה) באמצעות MLE, והקוד בוחר את ההתפלגות עם **סטטיסטי KS הקטן ביותר** (מבחן Kolmogorov-Smirnov). מי שנבחר משמש כסמפלר לאותה תופעה לאורך כל הסימולציה.

</div>

In [ ]:
samplers = fit_from_excel(EXCEL_PATH)
friends_sampler    = samplers['friends_interarrival']
main_stage_sampler = samplers['main_stage_duration']
print("\n-> Fitted distributions loaded as samplers.")

<div dir="rtl" style="text-align: right;">

## 2. ניתוח זמן חימום (Heating Time)

מריצים 30 רפליקציות ובוחנים את התנהגות המדדים בעזרת **ממוצע נע של Welch** (`Welch's moving average`). מטרת השלב: לזהות אחרי כמה ימי-סימולציה המערכת מגיעה למצב יציב — תקופה זו תוסר מהניתוח הסטטיסטי הראשי.

</div>

In [ ]:
warmup = WarmupSimulation(n_runs=30)
warmup.run()
warmup.plot_heating_time_data(warmup.daily_avg_queue_lengths, 'Average Queue Length')
warmup.plot_heating_time_data(warmup.daily_avg_satisfactions, 'Average Satisfaction')

<div dir="rtl" style="text-align: right;">

## 3. Data Points For Current State — תרחיש בסיס

מריצים את תרחיש הבסיס (Baseline) ל-20 רפליקציות בלתי תלויות. בכל ריצה משתמשים ב-seed שונה (1000, 1001, ... , 1019) כך שהריצות בלתי תלויות זו בזו אך משחזרות.

</div>

In [ ]:
def run_scenario(name, cfg, num_runs, friends_sampler, main_stage_sampler, base_seed):
    """Run `num_runs` replications of a scenario and return MultiRunStatistics."""
    multi = MultiRunStatistics(CONFIDENCE_LEVEL, RELATIVE_PRECISION)
    print(f"\n  Running '{name}' — {num_runs} replications ...")
    for i in range(num_runs):
        sim = Simulation(cfg,
                         friends_arrival_sampler=friends_sampler,
                         main_stage_duration_sampler=main_stage_sampler)
        s = sim.run(seed=base_seed + i)
        multi.add_run(s)
    print(f"  Done.")
    return multi

baseline_alt   = build_baseline()
baseline_stats = run_scenario('Baseline', baseline_alt.config, NUM_RUNS,
                              friends_sampler, main_stage_sampler, base_seed=1000)
print(baseline_stats.report())

<div dir="rtl" style="text-align: right;">

### בדיקת מספר ריצות נדרש (דיוק יחסי)

לכל מדד בודקים האם 20 רפליקציות מספיקות לדיוק יחסי של γ = 0.1 ברמת בטחון 0.9. הקריטריון:

$$\frac{\delta(\alpha,n)}{|\bar{x}|} \le \frac{\gamma}{1+\gamma}$$

כאשר $\delta(\alpha,n) = t_{n-1,1-\alpha/2} \cdot s/\sqrt{n}$. אם הקריטריון לא מתקיים, נחשב כמה רפליקציות נוספות נדרשות.

</div>

In [ ]:
def relative_accuracy_check(multi, kpi, gamma=0.10):
    """Check whether the current replication count meets relative precision `gamma`.

    Returns a dict with the mean, std, relative error, threshold gamma/(1+gamma),
    a boolean for whether the criterion is met, and the additional replication
    count needed if not.
    """
    values = multi._kpi_values(kpi)
    n = len(values)
    mean = stats_lib.mean(values)
    std  = stats_lib.stdev(values)
    alpha = 1.0 - multi.confidence_level
    t_crit = student_t.ppf(1 - alpha / 2, df=n-1)
    delta = t_crit * std / math.sqrt(n)
    rel_err   = delta / abs(mean) if mean else float('inf')
    threshold = gamma / (1 + gamma)
    meets     = rel_err <= threshold
    n_needed  = multi.required_replications(kpi, pilot_runs=n)
    return {
        'kpi': kpi, 'mean': mean, 'std': std, 'n': n,
        'relative_error': rel_err, 'threshold': threshold,
        'meets_criterion': meets, 'n_needed': n_needed,
    }

print(f"{'KPI':25s} {'mean':>14s} {'rel_err':>10s} {'γ/(1+γ)':>12s} {'meets?':>8s} {'n needed':>10s}")
print('-' * 82)
for kpi in KPIS:
    r = relative_accuracy_check(baseline_stats, kpi, gamma=RELATIVE_PRECISION)
    print(f"{r['kpi']:25s} {r['mean']:14.4f} {r['relative_error']:10.4f} "
          f"{r['threshold']:12.4f} {str(r['meets_criterion']):>8s} {r['n_needed']:>10d}")

<div dir="rtl" style="text-align: right;">

### רווחי סמך למדדי Baseline

רווח סמך t-Student לכל מדד ברמת בטחון 90%:

$$\bar{x}(n) \pm t_{n-1,1-\alpha/2} \cdot \frac{s(n)}{\sqrt{n}}$$

</div>

In [ ]:
print(f"{'KPI':25s} {'mean':>14s} {'CI lower':>14s} {'CI upper':>14s}")
print('-' * 70)
for kpi in KPIS:
    mean, lo, hi = baseline_stats.confidence_interval(kpi)
    print(f"{kpi:25s} {mean:14.4f} {lo:14.4f} {hi:14.4f}")

<div dir="rtl" style="text-align: right;">

## 4. Alternative Review — סקירת חלופות

שלוש חלופות שדרוג, כולן בתוך תקרת התקציב של 1,000,000 ₪. כל חלופה מורצת בנפרד עם seeds שונים (Combo_A: 11000+, Combo_B: 21000+, Combo_C: 31000+) — כך **הריצות בין החלופות בלתי תלויות**, וניתן יהיה להחיל מבחן Welch.

**שלוש החלופות**:
- **Combo_A** (950,000 ₪): צילום+ארט נוסף + שיווק + סריקת כרטיסים אוטומטית — *Overall Winner* (סכום-דירוגים נמוך ביותר על שלושת המדדים)
- **Combo_B** (1,000,000 ₪): שיווק + סריקת כרטיסים אוטומטית + שקית מתנה — *Revenue King* (הכנסה הכי גבוהה)
- **Combo_C** (650,000 ₪): להקות פופולריות + צילום+ארט נוסף + שקית מתנה — *Satisfaction King* (סיפוק הכי גבוה)

</div>

In [ ]:
combo_a_alt   = build_combo_a()
combo_a_stats = run_scenario(combo_a_alt.name, combo_a_alt.config, NUM_RUNS,
                             friends_sampler, main_stage_sampler, base_seed=11000)
print(combo_a_stats.report())

In [ ]:
combo_b_alt   = build_combo_b()
combo_b_stats = run_scenario(combo_b_alt.name, combo_b_alt.config, NUM_RUNS,
                             friends_sampler, main_stage_sampler, base_seed=21000)
print(combo_b_stats.report())

In [ ]:
combo_c_alt   = build_combo_c()
combo_c_stats = run_scenario(combo_c_alt.name, combo_c_alt.config, NUM_RUNS,
                             friends_sampler, main_stage_sampler, base_seed=31000)
print(combo_c_stats.report())

<div dir="rtl" style="text-align: right;">

## 5. Comparison of Alternatives By Welch

**הנחות מבחן Welch:**
- אין תלות בין ריצות סימולציה שמבוצעות במקביל.
- אין תלות בין הריצות השונות בכל חלופה בנפרד.
- לא ניתן להניח כי השונויות בין החלופות זהות.
- המדדים שנבדקים מתפלגים נורמלית.

*הנחת הנורמליות מתאפשרת מכיוון שמדובר במדדים שהינם ממוצעים, כך שניתן להניח נורמליות לפי משפט הגבול המרכזי. בנוסף, אין תלות בין הריצות השונות עבור החלופות מכיוון שהשתמשנו ב-`random.random` עם seed שונה לכל חלופה (Baseline: 1000+, Combo_A: 11000+, Combo_B: 21000+, Combo_C: 31000+).*

**נוסחת המבחן** — נחשב רווח סמך על ההפרש בין ממוצעי שתי החלופות:

$$\bar{x}_1(n_1) - \bar{x}_2(n_2) \pm t_{\hat{f},1-\alpha/2} \cdot \sqrt{\frac{s_1^2(n_1)}{n_1} + \frac{s_2^2(n_2)}{n_2}}$$

כאשר $\hat{f}$ נקבע לפי משוואת Welch-Satterthwaite:

$$\hat{f} = \frac{\left[\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}\right]^2}{\frac{(s_1^2/n_1)^2}{n_1-1} + \frac{(s_2^2/n_2)^2}{n_2-1}}$$

**פירוש**: אם 0 בתוך רווח הסמך — לא נוכל לקבוע אי-זו חלופה עדיפה. אם הרווח כולו חיובי או כולו שלילי — נכריע לטובת הצד המתאים.

</div>

In [ ]:
def print_welch_comparison(baseline, alternative, alt_name):
    """Print the Welch CI on the mean difference (baseline - alternative) for each KPI."""
    conf_pct = int(round(CONFIDENCE_LEVEL * 100))
    print(f"\n  --- {alt_name} vs Baseline (Welch's t-test) ---")
    for kpi in KPIS:
        r = baseline.welch_t_test(alternative, kpi)
        diff, lo, hi = r['diff'], r['ci_lower'], r['ci_upper']
        higher_is_better = KPI_HIGHER_IS_BETTER.get(kpi, True)
        print(f"    {kpi}")
        print(f"      Mean difference (baseline - {alt_name}): {diff:+.4f}")
        print(f"      {conf_pct}% Confidence Interval: [{lo:+.4f}, {hi:+.4f}]")
        if lo > 0:
            winner = 'Baseline' if higher_is_better else alt_name
            print(f"      0 outside CI -> {winner} significantly better")
        elif hi < 0:
            winner = alt_name if higher_is_better else 'Baseline'
            print(f"      0 outside CI -> {winner} significantly better")
        else:
            print("      0 inside CI -> cannot determine which is better")

print_welch_comparison(baseline_stats, combo_a_stats, 'Combo_A')
print_welch_comparison(baseline_stats, combo_b_stats, 'Combo_B')
print_welch_comparison(baseline_stats, combo_c_stats, 'Combo_C')

<div dir="rtl" style="text-align: right;">

## 6. ניתוח הפרויקט — מסקנות והמלצות

הפרויקט מדמה באמצעות סימולציה בתכנות אירועים את ההתנהלות של פסטיבל מוזיקה בן יומיים, מעקב אחר שלושה מדדי-על — **סיפוק המבקרים**, **ההכנסה הכוללת** ו**אורך התור הממוצע**.

במסגרת התקציב של 1,000,000 ₪ בחנו שלוש חלופות:

1. **Combo_A** (950,000 ₪) — מתמקדת בהסרת צוואר-הבקבוק של הכניסה ובהקטנת תורי שירות.
2. **Combo_B** (1,000,000 ₪) — מתמקדת ב-ROI הכלכלי (שיווק + יעילות כניסה + שביעות רצון התחלתית).
3. **Combo_C** (650,000 ₪) — הזולה ביותר; מתמקדת בעליית סיפוק (להקות פופולריות + שקית מתנה).

### מסקנות לפי תוצאות המבחן

*(מלא לפי מה שיתקבל בריצה — באיזה מדד 0 בתוך הרווח, באיזה הוא מחוץ ולאיזה צד.)*

**Combo_A**: ...

**Combo_B**: ...

**Combo_C**: ...

### המלצה סופית

...

</div>